In [1]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [2]:
df = pd.read_csv('covid_toy.csv')

In [3]:
df.sample(4)

,age,gender,fever,cough,city,has_covid
58,23,Male,98.0,Strong,Mumbai,Yes
61,81,Female,98.0,Strong,Mumbai,No
92,82,Female,102.0,Strong,Kolkata,No
68,54,Female,104.0,Strong,Kolkata,No


In [4]:
df['cough'].value_counts()

cough
Mild      62
Strong    38
Name: count, dtype: int64

In [5]:
df['city'].value_counts()

city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

In [6]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [7]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['has_covid']),df['has_covid'],test_size=0.2)

In [8]:
X_train.sample(3)

,age,gender,fever,cough,city
29,34,Female,NaN,Strong,Mumbai
35,82,Female,102.0,Strong,Bangalore
65,69,Female,102.0,Mild,Bangalore


# Without use Column Transformer

In [10]:
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])

X_test_fever = si.fit_transform(X_test[['fever']])

X_train_fever.shape

(80, 1)

In [11]:
oe = OrdinalEncoder(categories=[['Mild','Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

X_test_cough = oe.fit_transform(X_test[['cough']])

X_train_cough.shape

(80, 1)

In [12]:
ohe = OneHotEncoder(drop='first',sparse_output=False)

X_train_gen_city = ohe.fit_transform(X_train[['gender','city']])

X_test_gen_city = ohe.fit_transform(X_test[['gender','city']])

X_train_gen_city.shape

(80, 4)

In [13]:
X_train_age = X_train.drop(columns=['gender','fever','cough','city']).values

X_test_age = X_test.drop(columns=['gender','fever','cough','city']).values

X_train_age.shape

(80, 1)

In [14]:
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gen_city,X_train_cough),axis=1)
X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gen_city,X_test_cough),axis=1)

X_train_transformed.shape

(80, 7)

# With ColumnTransformer

In [18]:
from sklearn.compose import ColumnTransformer

trans = ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])
],remainder='passthrough')

In [22]:
trans.fit_transform(X_train).shape

(80, 7)

In [23]:
X_train_transformed.shape

(80, 7)